# 07. Трансформации данных — <span style="color:#C9A0DC">torchvision.transforms.v2</span>

Видео: [7. Трансформация данных. Transforms и V2](https://www.youtube.com/watch?v=9ZnQch_O6xE&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=8)

## О чём урок

В уроках 5–6 ты писал <span style="color:#7CDBA0">Dataset</span> и сам превращал картинку в тензор прямо в `__getitem__`: `np.array` → деление на 255 → `unsqueeze(0)`. Это работает, но обработка «зашита» внутрь класса.

Урок 7 выносит обработку в отдельный объект — <span style="color:#E6C07B">transform</span>. Это функция (или цепочка функций), которую датасет вызывает для каждой картинки при чтении:

```text
файл на диске
  → Image.open          (PIL-картинка)
  → transform(...)      (тензор для сети)
  → DataLoader          (батч)
  → модель
```

В видео показаны два модуля: старый `torchvision.transforms` и новый `torchvision.transforms.v2`. **Здесь только v2** — так пишут новый код, и автор дальше по курсу тоже использует v2.

## Рабочая цепочка, которую разберём

```text
PIL
  → v2.ToImage()                          # тензор формы (C, H, W), пока uint8 0…255
  → (аугментации: flip и т.п.)            # только для train, не в этом ноутбуке
  → v2.ToDtype(torch.float32, scale=True) # float, значения в [0, 1]
  → v2.Normalize(mean, std)               # (x - mean) / std, значения около нуля
```

## План

| блок | что делаешь | зачем |
|:---|:---|:---|
| 0 | создаёшь тестовые данные | картинки для экспериментов |
| 1 | PIL → тензор через `ToImage` + `ToDtype`; `permute` против `reshape` | понять формат, который ждёт PyTorch |
| 2 | `Normalize` + `Compose` | стандартизация входа и сборка цепочки |
| 3 | transform внутри своего `Dataset` и `ImageFolder` | как это подключается к данным из уроков 5–6 |
| 4 | свой transform через `__call__` | что делать, если готового шага нет |
| 5 | итог: что сделали и зачем | |

**Как решать.**

> **Конспект:** зачем нормализовать вход — [`d.4.3`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.3 Регуляризация в Deep Learning/d.4.3 Регуляризация в Deep Learning.md) **§1**; пример на цифрах — [`d.4.3.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.3 Регуляризация в Deep Learning/d.4.3.1 Пример — нормализация и один шаг обучения.md). Карта — [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md).
 Замени `...` на код → запусти ячейку → запусти проверку под ней. Если проверка упала, сначала читай текст ошибки. Решения — в `07_transforms_v2_решения.ipynb`, только после своей попытки.


In [ ]:
import json
import os

import numpy as np
import torch
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"


print("torch", torch.__version__, "| v2 ok")


## 0. Данные

Ячейка ниже создаёт папку `_data/` рядом с ноутбуком (если её ещё нет):

```text
_data/
├── sample_rgb.png     # цветная картинка 32×32: чёрный фон, красный и зелёный квадраты
├── fruits/train/      # серые картинки 8×8 по папкам-классам apple / pear / plum
└── squares/           # серые 64×64 со светлым квадратом + labels.json с [cx, cy]
```

| данные | где пригодятся |
|:---|:---|
| `sample_rgb.png` | блоки 1, 2, 4: на цветной картинке видно три канала |
| `fruits/` | блок 3: свой датасет и `ImageFolder` для классификации |
| `squares/` | блок 3: датасет для регрессии из урока 6 |

Ячейка также открывает `sample_rgb.png` двумя способами и печатает:
- `pil_rgb` — PIL-картинка (так её отдаёт `Image.open`);
- `arr_hwc` — та же картинка как numpy-массив формы `(32, 32, 3)`, то есть **HWC**: высота, ширина, каналы. Числа целые, от 0 до 255.

**Ожидаемый вывод:**

```text
PIL: RGB (32, 32)
numpy HWC: (32, 32, 3) uint8 0 255
```


In [ ]:
DATA = os.path.join(".", "_data")
SAMPLE_RGB = os.path.join(DATA, "sample_rgb.png")
FRUITS_TRAIN = os.path.join(DATA, "fruits", "train")
SQUARES_DIR = os.path.join(DATA, "squares")
LABELS_PATH = os.path.join(SQUARES_DIR, "labels.json")


def _make_sample_rgb():
    os.makedirs(DATA, exist_ok=True)
    img = Image.new("RGB", (32, 32), (0, 0, 0))
    draw = ImageDraw.Draw(img)
    draw.rectangle([4, 4, 15, 15], fill=(255, 40, 40))
    draw.rectangle([16, 16, 27, 27], fill=(40, 255, 40))
    img.save(SAMPLE_RGB)


def _write_gray(path, value):
    Image.fromarray(np.full((8, 8), value, dtype=np.uint8), mode="L").save(path)


def _make_fruits():
    layout = {"train": {"apple": 12, "pear": 10, "plum": 3}}
    bright = {"apple": 40, "pear": 120, "plum": 200}
    for split, classes in layout.items():
        for cls, n in classes.items():
            d = os.path.join(DATA, "fruits", split, cls)
            os.makedirs(d, exist_ok=True)
            for i in range(n):
                _write_gray(os.path.join(d, f"{cls}_{i:03d}.png"), bright[cls])


def _make_squares(n=80, size=64, side=15, seed=0):
    os.makedirs(SQUARES_DIR, exist_ok=True)
    rng = np.random.default_rng(seed)
    half = side // 2
    labels = {}
    for i in range(n):
        name = f"img_{i:03d}.png"
        cx = int(rng.integers(half, size - half))
        cy = int(rng.integers(half, size - half))
        img = Image.new("L", (size, size), 30)
        draw = ImageDraw.Draw(img)
        draw.rectangle(
            [cx - half, cy - half, cx - half + side - 1, cy - half + side - 1], fill=220
        )
        img.save(os.path.join(SQUARES_DIR, name))
        labels[name] = [float(cx), float(cy)]
    with open(LABELS_PATH, "w", encoding="utf-8") as f:
        json.dump(labels, f)


if not os.path.isfile(SAMPLE_RGB):
    _make_sample_rgb()
if not os.path.isdir(FRUITS_TRAIN):
    _make_fruits()
if not os.path.isfile(LABELS_PATH):
    _make_squares()

pil_rgb = Image.open(SAMPLE_RGB)
arr_hwc = np.array(pil_rgb)
print("PIL:", pil_rgb.mode, pil_rgb.size)
print("numpy HWC:", arr_hwc.shape, arr_hwc.dtype, int(arr_hwc.min()), int(arr_hwc.max()))


## 1. Из PIL в тензор: <span style="color:#C9A0DC">ToImage</span> + <span style="color:#C9A0DC">ToDtype</span> (видео ~1:34–7:03)

### Проблема

PIL и numpy хранят картинку как **HWC** — `(высота, ширина, каналы)`. Каждый пиксель лежит целиком: `[R, G, B]`.

PyTorch ждёт **CHW** — `(каналы, высота, ширина)`. Сначала все красные значения, потом все зелёные, потом все синие. Плюс сети нужны дробные числа, а не целые 0…255.

### Решение — два шага v2

| шаг | что делает | форма и тип после |
|:---|:---|:---|
| `v2.ToImage()` | PIL/numpy → тензор, оси переставлены в CHW | `(3, 32, 32)`, `uint8`, 0…255 |
| `v2.ToDtype(torch.float32, scale=True)` | в дробные числа; `scale=True` делит на 255 | `(3, 32, 32)`, `float32`, 0…1 |

Вместе они заменяют устаревший `ToTensor` из старого модуля.

Если `scale=False`, тип станет `float32`, но значения останутся 0…255.

### Задание 1.1

Собери `v2.Compose` из этих двух шагов и примени к `pil_rgb`.

**Ожидаемый вывод:**

```text
torch.Size([3, 32, 32]) torch.float32 0.0 1.0
```

Если напечатать сам `x`, увидишь почти одни нули. Это не ошибка: PyTorch показывает только углы большого тензора, а углы картинки чёрные. Цветные квадраты в середине скрыты за `...`. Об этом говорит `max = 1.0`.


In [ ]:
# 1.1
to_float = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ]
)
x = to_float(pil_rgb)
print(x.shape, x.dtype, float(x.min()), float(x.max()))


In [ ]:
need_tensor(x, "x")
need(tuple(x.shape) == (3, 32, 32), f"shape={tuple(x.shape)}")
need(x.dtype == torch.float32, f"dtype={x.dtype}")
need(0.0 <= float(x.min()) and float(x.max()) <= 1.0, "scale=True → [0, 1]")
only = v2.ToImage()(pil_rgb)
need(only.dtype == torch.uint8 and int(only.max()) > 1, "один ToImage ещё uint8 0..255")
print("1.1 — верно")


### Задание 1.2 — <span style="color:#C9A0DC">permute</span> против <span style="color:#C9A0DC">reshape</span> (видео ~2:37–3:41)

Частая ошибка: переставить каналы через `reshape(3, H, W)`. Форма получится правильная, а картинка сломается.

**Почему.** В памяти тензор — это одна лента чисел, а форма — только инструкция, как её читать.

Мини-картинка из двух пикселей, HWC `(1, 2, 3)`:

```text
пиксель 1 = (R=1, G=2, B=3), пиксель 2 = (R=4, G=5, B=6)
лента в памяти: 1 2 3 4 5 6
```

| операция | что делает | результат «по каналам» |
|:---|:---|:---|
| `permute(2, 0, 1)` | меняет местами оси: канал становится первой | R: `[1, 4]`, G: `[2, 5]`, B: `[3, 6]` — правильно |
| `reshape(3, 1, 2)` | режет ту же ленту на куски по 2, ничего не переставляя | «R»: `[1, 2]`, «G»: `[3, 4]`, «B»: `[5, 6]` — каналы перемешаны |

**Правило:**
- `reshape` — когда меняешь только форму, а порядок чисел уже правильный (например, вытянуть картинку `(1, 28, 28)` в вектор `(784,)`);
- `permute` — когда нужно, чтобы другая ось стала первой (HWC → CHW).

Сделай из `arr_hwc` два тензора: `ok` через `permute(2, 0, 1)` и `bad` через `reshape(3, 32, 32)`. Проверка поделит их на 255 и сравнит с `x` из 1.1.

**Ожидаемый вывод:**

```text
ok совпал с x? True
bad совпал с x? False
```


In [ ]:
# 1.2
ok = torch.from_numpy(arr_hwc).permute(2, 0, 1)
bad = torch.from_numpy(arr_hwc).reshape(3, 32, 32)
print("ok совпал с x?", torch.allclose(ok.float() / 255.0, x, atol=1e-5))
print("bad совпал с x?", torch.allclose(bad.float() / 255.0, x, atol=1e-2))


In [ ]:
need(tuple(ok.shape) == (3, 32, 32) and tuple(bad.shape) == (3, 32, 32), "формы (3,32,32)")
need(torch.allclose(ok.float() / 255.0, x, atol=1e-5), "permute правильный")
need(not torch.allclose(bad.float() / 255.0, x, atol=1e-2), "reshape ломает каналы")
print("1.2 — верно")


## 2. <span style="color:#C9A0DC">Normalize</span> + <span style="color:#C9A0DC">Compose</span> (видео ~3:41–4:45)

### Что делает Normalize

Для каждого канала отдельно:

$$
x' = \frac{x - \text{mean}}{\text{std}}
$$

Это та же стандартизация, что `StandardScaler` в классическом ML ([`d.4.3`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.3 Регуляризация в Deep Learning/d.4.3 Регуляризация в Deep Learning.md) §1 (+ пример [`d.4.3.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.3 Регуляризация в Deep Learning/d.4.3.1 Пример — нормализация и один шаг обучения.md)); классика — `d.2.2` / StandardScaler). Центр данных переезжает в ноль, разброс становится сопоставимым. Градиентному спуску так проще: один шаг обучения подходит всем весам.

- **mean** — среднее значение;
- **std** — стандартное отклонение, «типичный разброс вокруг mean». Всегда ≥ 0.
- На вход `Normalize` нужен уже **float**-тензор, поэтому он стоит после `ToDtype`.
- Чисел в `mean` и `std` столько же, сколько каналов: для RGB по три, для серой картинки по одному.

### Почему `mean = std = 0.5`

Эти числа выбраны для удобства, а не посчитаны по данным. Они переводят отрезок `[0, 1]` ровно в `[-1, 1]`:

| пиксель после `ToDtype` | $(x - 0.5) / 0.5$ |
|:---|:---|
| 0 (чёрный) | −1 |
| 0.5 | 0 |
| 1 (белый) | 1 |

Честный вариант — посчитать mean и std по пикселям **train**-картинок и применять те же числа к val и test. Для обучения подходят оба варианта.

### Compose

`v2.Compose([шаг1, шаг2, ...])` применяет шаги по очереди. Это аналог `Pipeline` из sklearn.

### Задание 2.1

Собери цепочку `ToImage → ToDtype(float32, scale=True) → Normalize(0.5 на 3 канала)` и примени к `pil_rgb`.

**Ожидаемый вывод:**

```text
torch.Size([3, 32, 32]) torch.float32 -1.0 1.0
```

Проверка сравнит результат с ручным расчётом `(x - 0.5) / 0.5`.


In [ ]:
# 2.1
tf = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
)
y = tf(pil_rgb)
print(y.shape, y.dtype, float(y.min()), float(y.max()))


In [ ]:
need_tensor(y, "y")
need(tuple(y.shape) == (3, 32, 32) and y.dtype == torch.float32, f"{tuple(y.shape)} {y.dtype}")
need(float(y.min()) < 0.0, "после Normalize тёмные пиксели < 0")
need(torch.allclose(y, (x - 0.5) / 0.5, atol=1e-6), "должно совпасть с ручным (x-0.5)/0.5")
print("2.1 — верно")


## 3. Transform внутри <span style="color:#7CDBA0">Dataset</span> / <span style="color:#E6C07B">ImageFolder</span> (видео ~7:03–10:10)

### Идея

Transform передают **один раз** в конструктор датасета. Потом он срабатывает **при каждом** `ds[i]` — и когда ты сам берёшь пример, и когда `DataLoader` собирает батч.

```python
def __getitem__(self, idx):
    img = Image.open(path)          # PIL
    if self.transform is not None:
        img = self.transform(img)   # здесь PIL превращается в тензор
    return img, label
```

**Зачем так, а не руками внутри класса.** Датасет отвечает за то, *где взять* картинку и метку, а transform — за то, *как её обработать*. Один и тот же класс можно создать с разными transform: на train с аугментациями, на val/test без них.

### Задание 3.1 — свой датасет фруктов

Класс почти готов: `__init__` собирает пары `(путь, номер класса)`, как `FruitsDataset` из урока 5. Допиши в `__getitem__` применение transform и собери `tf_gray`: `ToImage → ToDtype(scale=True) → Normalize((0.5,), (0.5,))`.

`(0.5,)` — кортеж из одного числа, потому что канал у серой картинки один. Запятая обязательна: `(0.5)` — это просто число.

**Ожидаемый вывод:**

```text
torch.Size([1, 8, 8]) torch.float32 -0.686... 0
```

Последнее число — метка класса (`apple` = 0).


In [ ]:
# 3.1
class TinyGrayDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.classes = sorted(
            d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))
        )
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples = []
        for c in self.classes:
            d = os.path.join(root, c)
            for name in sorted(os.listdir(d)):
                if name.endswith(".png"):
                    self.samples.append((os.path.join(d, name), self.class_to_idx[c]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path)
        if self.transform is not None:
            img = self.transform(img)
        return img, label


tf_gray = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=(0.5,), std=(0.5,)),
    ]
)
train_ds = TinyGrayDataset(FRUITS_TRAIN, transform=tf_gray)
x3, y3 = train_ds[0]
print(x3.shape, x3.dtype, float(x3.min()), y3)


In [ ]:
need_tensor(x3, "x3")
need(tuple(x3.shape) == (1, 8, 8), f"x3={tuple(x3.shape)}")
need(x3.dtype == torch.float32 and float(x3.min()) < 0.0, "float + Normalize")
need(isinstance(y3, int), f"label={type(y3)}")
print("3.1 — верно")


### Задание 3.2 — <span style="color:#E6C07B">ImageFolder</span> и <span style="color:#C9A0DC">Grayscale</span> (видео ~9:07)

`ImageFolder` из урока 6 тоже принимает `transform=`. Но у него есть нюанс: даже серые PNG он часто отдаёт как **3 одинаковых канала**.

Модель под серый вход ждёт форму `(1, H, W)`. Если придёт `(3, H, W)`, будет ошибка формы или лишние веса на повторяющиеся каналы. Лечится шагом `v2.Grayscale(num_output_channels=1)` в начале цепочки.

Твой `TinyGrayDataset` из 3.1 этим не страдает: он открывает PNG в родном режиме, а серый PNG — это один канал.

| картинка | каналов | нужен `Grayscale(1)`? |
|:---|:---|:---|
| цветная (RGB) | 3 | нет |
| серая (оттенки от чёрного к белому) | 1 | да, если читаешь через `ImageFolder` |

Собери две цепочки:
- `tf_a`: `ToImage → ToDtype(scale=True)` — без `Grayscale`;
- `tf_b`: `Grayscale(1) → ToImage → ToDtype(scale=True)`.

**Ожидаемый вывод:**

```text
без: torch.Size([3, 8, 8]) с Grayscale: torch.Size([1, 8, 8])
```


In [ ]:
# 3.2
tf_a = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
tf_b = v2.Compose(
    [
        v2.Grayscale(num_output_channels=1),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ]
)
xa, _ = ImageFolder(FRUITS_TRAIN, transform=tf_a)[0]
xb, _ = ImageFolder(FRUITS_TRAIN, transform=tf_b)[0]
print("без:", xa.shape, "с Grayscale:", xb.shape)


In [ ]:
need(tuple(xa.shape) == (3, 8, 8), f"без Grayscale: {tuple(xa.shape)}")
need(tuple(xb.shape) == (1, 8, 8), f"с Grayscale: {tuple(xb.shape)}")
print("3.2 — верно")


### Задание 3.3 — регрессия: <span style="color:#7CDBA0">SquaresDataset</span> с transform (видео ~9:37–10:10)

Это `SquaresDataset` из урока 6, но вместо ручного `np.array → / 255 → unsqueeze(0)` картинку обрабатывает transform.

| | урок 6 | урок 7 |
|:---|:---|:---|
| перевод картинки в тензор | руками в `__getitem__` | `transform`, переданный в конструктор |
| значения пикселей | `[0, 1]` | `[-1, 1]` (добавился `Normalize`) |
| координаты `[cx, cy]` | `float32`, форма `(2,)` | без изменений |

Важно: transform применяется **только к картинке**, координаты он не трогает. Поэтому здесь нельзя добавлять геометрические аугментации (отражение, поворот, сдвиг): квадрат сдвинется, а `cx, cy` останутся старыми. Яркость или шум менять можно — положение квадрата от них не зависит.

**Ожидаемый вывод:**

```text
torch.Size([1, 64, 64]) torch.Size([2]) torch.Size([8, 1, 64, 64]) torch.Size([8, 2])
```

Первые две формы — один пример, последние две — батч из 8: к каждой форме спереди добавилась ось батча.


In [ ]:
# 3.3
class SquaresDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.names = sorted(n for n in os.listdir(root) if n.endswith(".png"))
        with open(os.path.join(root, "labels.json"), encoding="utf-8") as f:
            self.labels = json.load(f)

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        img = Image.open(os.path.join(self.root, name))
        if self.transform is not None:
            img = self.transform(img)
        coords = torch.tensor(self.labels[name], dtype=torch.float32)
        return img, coords


tf_sq = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=(0.5,), std=(0.5,)),
    ]
)
sq = SquaresDataset(SQUARES_DIR, transform=tf_sq)
sx, sy = sq[0]
bx, by = next(iter(DataLoader(sq, batch_size=8)))
print(sx.shape, sy.shape, bx.shape, by.shape)


In [ ]:
need(tuple(sx.shape) == (1, 64, 64) and tuple(sy.shape) == (2,), f"{tuple(sx.shape)} {tuple(sy.shape)}")
need(tuple(bx.shape) == (8, 1, 64, 64) and tuple(by.shape) == (8, 2), f"batch {tuple(bx.shape)} {tuple(by.shape)}")
print("3.3 — верно")


## 4. Свой transform (видео ~10:42–12:18)

Если готового шага в `v2` нет, его пишут сами. Transform — это любой объект, который можно вызвать как функцию: `img = tf(img)`.

Самый простой способ — класс с методом `__call__`:

```python
class MyStep:
    def __init__(self, ...):      # запомнить параметры
        ...
    def __call__(self, img):      # что делать с одной картинкой
        return ...
```

Объект такого класса можно положить в `v2.Compose` рядом со встроенными шагами.

Автор в видео показывает ещё вариант через наследование от `nn.Module` с методом `forward`. Для предобработки данных достаточно `__call__`; `nn.Module` подробно будет в видео 9.

### Задание 4.1

Напиши свой `Normalize`: в `__init__` запомни `mean` и `std`, в `__call__` верни `(img - mean) / std`.

`mean_t` и `std_t` уже даны формы `(3, 1, 1)`, а не `(3,)`. Так при вычитании из картинки `(3, 32, 32)` каждый канал получает своё число: PyTorch растягивает `(3, 1, 1)` на всю высоту и ширину (трансляция, broadcasting).

**Ожидаемый вывод:**

```text
True
```

Это значит, что твой класс даёт ровно тот же тензор, что `v2.Normalize`.


In [ ]:
# 4.1
class CallNormalize:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, img):
        return (img - self.mean) / self.std


mean_t = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)
std_t = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)

built_in = v2.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))(x)
mine = CallNormalize(mean_t, std_t)(x)
print(torch.equal(built_in, mine))


In [ ]:
need(torch.equal(built_in, mine), "свой Normalize должен совпасть с v2.Normalize")
print("4.1 — верно")


## 5. Что мы сделали в этом уроке

1. **Перевели картинку в формат PyTorch.** `ToImage` переставляет оси HWC → CHW, `ToDtype(float32, scale=True)` даёт дробные числа в `[0, 1]`. Раньше ты делал это руками в `__getitem__`.
2. **Разобрали ловушку `reshape`.** `reshape` меняет только форму и не переставляет оси. Для смены порядка осей нужен `permute`.
3. **Стандартизировали вход через `Normalize`.** Формула $(x - \text{mean}) / \text{std}$, как `StandardScaler`. С `mean = std = 0.5` пиксели переходят из `[0, 1]` в `[-1, 1]`.
4. **Собрали цепочку через `Compose`.** Несколько шагов в одном объекте, как `Pipeline` в sklearn.
5. **Подключили transform к датасетам.** Свой датасет фруктов, `ImageFolder` (с `Grayscale` для серых картинок) и `SquaresDataset` для регрессии.
6. **Написали свой transform** через `__call__` и проверили, что он совпадает со встроенным.

## Зачем это знать

- **Без этого данные не попадут в сеть.** Любая модель на картинках начинается с цепочки «PIL → тензор → нормализация». В следующих видео (модель, цикл обучения) этот шаг будет в каждом проекте, в том числе в твоём `project/fashion_mnist.ipynb`.
- **Нормализация влияет на обучение.** Вход около нуля и в одном масштабе — градиентный спуск идёт ровнее, сеть учится быстрее и стабильнее.
- **Формы тензоров — главный источник ошибок.** `permute`, `reshape`, количество каналов, `Grayscale` — то, на чём чаще всего ломается код с ошибкой `shape mismatch`.
- **Идея переносится за пределы картинок.** В рекомендациях и табличных данных та же схема: `Dataset` читает один пример, а предобработка (стандартизация чисел, перевод id в индексы) — отдельный шаг. `Compose` ≈ `Pipeline`, `Normalize` ≈ `StandardScaler`, правило «mean и std только по train» то же самое.
- **Разделение train и eval.** Потому что transform отделён от датасета, на train можно добавить случайные аугментации, а на val/test оставить только детерминированную обработку. Это будет в разделе «Видео 7» проекта.

## Шпаргалка

```text
PIL → ToImage → [аугментации, только train] → ToDtype(float32, scale=True) → Normalize → батч
```

| инструмент | роль |
|:---|:---|
| `v2.Compose([...])` | цепочка шагов |
| `v2.ToImage()` | PIL/numpy → тензор CHW (uint8) |
| `v2.ToDtype(torch.float32, scale=True)` | в float, `[0, 1]` |
| `v2.Normalize(mean, std)` | `(x - mean) / std`, по числу каналов |
| `v2.Grayscale(1)` | один канал для серых картинок в `ImageFolder` |
| `permute` / `reshape` | смена порядка осей / смена только формы |
| класс с `__call__` | свой шаг предобработки |

**Дальше:** раздел «Видео 7» в `project/fashion_mnist.ipynb`, потом видео 8 — функции активации и потерь.
